In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from pylab import rcParams
import seaborn as sns
import warnings
import sys
import os

In [ ]:
# Constants

colors = ["#007CF3", "#1DB0D4"]
SEED = 42

In [ ]:
sns.set_style('darkgrid')
warnings.filterwarnings("ignore")
%matplotlib inline

sys.path.append(os.path.abspath(".."))

np.random.seed(SEED)

In [ ]:
from load_data import load_data

In [ ]:
df = load_data('./Train.csv')

In [ ]:
df.head()

In [ ]:
df.describe().T

In [ ]:
print(f'{"#":<5} {"Columnn":<10} {"Dtype":>12} {"Missing":>14} {"Missing (%)":>15}')
for j, i, k, d in zip(range(len(df.columns)), df.isnull().sum(), df.columns, df.dtypes):
    print(f"{j:<5} {k:<15} | {str(d):<10} | {i:>8} | {i / df.shape[0] * 100:6.2f}%")


In [ ]:
df.duplicated().sum()

In [ ]:
churn_counts = df['CHURN'].value_counts().sort_index()

plt.figure(figsize=(10,8))
plt.pie(
    churn_counts,
    labels=churn_counts.index,
    autopct='%1.1f%%',
    startangle=270,
    counterclock=False,
    colors=colors,
    explode=[0.005] * len(churn_counts),
)

plt.title("Distribution of Churn", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))
df.hist(bins=30, ax=ax, color=colors[0], histtype='bar')
ax.legend(loc='upper right', fontsize=12)
fig.tight_layout()
fig.show()

## Feature Engineering

In [ ]:
df.select_dtypes(object).nunique()

In [ ]:
df['REGION'] = df['REGION'].fillna('Unknown')

In [ ]:
df['TENURE'].unique()

In [ ]:
mapping = {
    'D 3-6 month': 3,
    'E 6-9 month': 6,
    'F 9-12 month': 9,
    'G 12-15 month': 12,
    'H 15-18 month': 15,
    'I 18-21 month': 18,
    'J 21-24 month': 21,
    'K > 24 month': 24
}

df['TENURE'] = df['TENURE'].map(mapping)

In [ ]:
to_drop = ['user_id', 'MRG']
df = df.drop(columns=to_drop)

In [ ]:
for column in df.select_dtypes(['float64', 'int64']):
    df[column] = df[column].fillna(df[column].median())


In [ ]:
df['TOP_PACK'].unique()

In [ ]:
df['TOP_PACK'] = df['TOP_PACK'].replace(np.nan, "Unknown")
value_counts = df['TOP_PACK'].value_counts()
rare_threshold = 100
df['TOP_PACK_CLEAN'] = df['TOP_PACK'].apply(
    lambda x: x if value_counts[x] > rare_threshold else "OTHER"
)


In [ ]:
df['TOP_PACK_CLEAN'] = (
    df['TOP_PACK_CLEAN']
    .str.lower()
    .str.replace(r'[\s_]+', ' ', regex=True)
    .str.strip()
)


In [ ]:
import re

def extract_price(text):
    match = re.search(r'(\d+)\s*f', text)
    return int(match.group(1)) if match else None

def extract_duration(text):
    match = re.search(r'(\d+)\s*(d|day|days|h|hour|hours|month)', text)
    return match.group(0) if match else None

def extract_type(text):
    if "data" in text: return "data"
    if "on-net" in text or "onn" in text: return "onnet"
    if "all-net" in text or "allnet" in text: return "allnet"
    if "mixt" in text: return "mixt"
    if "wifi" in text: return "wifi"
    if "internat" in text: return "internat"
    return "other"

df['PACK_PRICE'] = df['TOP_PACK_CLEAN'].apply(extract_price)
df['PACK_DURATION'] = df['TOP_PACK_CLEAN'].apply(extract_duration)
df['PACK_TYPE'] = df['TOP_PACK_CLEAN'].apply(extract_type)


In [ ]:
df.head()

In [ ]:
df['PACK_DURATION_CLEAN'] = (
    df['PACK_DURATION']
    .astype(str)
    .str.lower()
    .str.replace(r'\s+', '', regex=True)
)


In [ ]:
import numpy as np
import re

def duration_to_hours(val):
    if val is None or val.lower() == 'none' or val == 'nan':
        return np.nan

    match = re.match(r'(\d+)([dh])', val)
    if match:
        num, unit = match.groups()
        num = int(num)
        if unit == 'd':
            return num * 24
        elif unit == 'h':
            return num
    return np.nan

df['PACK_DURATION_HOURS'] = df['PACK_DURATION_CLEAN'].apply(duration_to_hours)


In [ ]:
to_drop = ['TOP_PACK', 'TOP_PACK_CLEAN', 'PACK_DURATION', 'PACK_DURATION_CLEAN']

df = df.drop(columns=to_drop)

In [ ]:
for column in df.select_dtypes(['float64', 'int64']):
    df[column] = df[column].fillna(df[column].median())

In [ ]:
df.isna().sum()

In [ ]:
df.head()

In [ ]:
df.REGION.nunique()

In [ ]:
df.PACK_TYPE.nunique()

In [ ]:
region_dummies = pd.get_dummies(df["REGION"], prefix="REGION")
pack_type_dummies = pd.get_dummies(df['PACK_TYPE'], prefix='PACK_TYPE')

In [ ]:
df = pd.concat(
    [
      df,
      region_dummies,
      pack_type_dummies
    ],
    axis=1
)

df

In [ ]:
df.isna().sum()

In [ ]:
bool_cols = df.select_dtypes('bool').columns

In [ ]:
for i in bool_cols:
    df[i] = df[i].astype(int)

In [ ]:
to_drop = ['REGION', 'PACK_TYPE']

df = df.drop(columns=to_drop)

In [ ]:
df.info()

In [ ]:
# Split Data into Features and Label

X = df.drop(columns=['CHURN'])
y = df['CHURN']

In [ ]:
X.head()

In [ ]:
X.columns

In [ ]:
scaler = StandardScaler()

feature_to_scale = ['MONTANT', 'REVENUE', 'ARPU_SEGMENT', 'DATA_VOLUME', 'PACK_PRICE', 'ON_NET']

scaler = ColumnTransformer(
    transformers=[("num", StandardScaler(), feature_to_scale)],
    remainder="passthrough"
)

X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=feature_to_scale + [col for col in X.columns if col not in feature_to_scale])


In [ ]:
X.head()